# MCP 실습 — 서버 셋을 붙인 보안 분석 에이전트

OpenAI Agents SDK 에이전트에 **MCP 서버 세 개**를 붙여 보안 공지를 대조하고,
결과를 **샌드박스 안에만** 저장합니다.

## 하는 일

```
사내 자산 목록(로컬)  +  벤더 공지(인터넷)  +  CVE 조회(사내 공용 서버)
                          ↓
                 sandbox/output/report.md
```

## 세 가지 MCP 연결 — 무엇이 다른가

표에서 **가로와 세로가 다른 축**이라는 점이 이 노트북의 핵심입니다.

| | 서버가 도는 곳 | 리소스가 있는 곳 | 전송 방식 |
|---|---|---|---|
| **MCP 서버 1** | 내 PC (자식 프로세스) | **내 디스크** | stdio |
| **MCP 서버 2** | 내 PC (자식 프로세스) | **인터넷** | stdio |
| **MCP 클라이언트 1** | **다른 호스트** | 그 호스트 | HTTP |

서버가 로컬에서 돈다고 리소스까지 로컬인 것이 아닙니다.
`fetch` 서버는 내 PC에서 돌지만 인터넷을 봅니다. 이 구분이 보안 설계의 출발점입니다.

## 준비물

```bash
uv add openai-agents mcp
npx -y @playwright/mcp@latest install-browser chrome-for-testing   # 최초 1회
```

`npx`(Node.js)와 `uvx`가 필요합니다. MCP 서버들이 그 명령으로 실행됩니다.

> 실행 순서: 위에서부터 차례대로. `mcp/` 폴더에서 실행하세요.

## Step 0. 준비

In [1]:
import asyncio
import functools
import http.server
import shutil
import socketserver
import subprocess
import sys
import threading
import time
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, MCPServerStreamableHttp

load_dotenv(override=True)

MODEL = "gpt-5.4-mini"
SANDBOX = Path("sandbox_agent").resolve()   # 에이전트가 손댈 수 있는 유일한 폴더
                                           # (다른 노트북의 sandbox/ 와 겹치지 않게 따로 둔다)
print("모델   :", MODEL)
print("샌드박스:", SANDBOX)

모델   : gpt-5.4-mini
샌드박스: D:\ai_edu_main\llm_edu\mcp\sandbox_agent


### 주피터에서 꼭 필요한 한 가지 손질

MCP의 stdio 서버는 **자식 프로세스**로 뜹니다. 그런데 노트북에서 그냥 연결하면 이렇게 실패합니다.

```
io.UnsupportedOperation: fileno
```

MCP 라이브러리는 서버의 오류 출력을 `sys.stderr`로 보내는데,
**주피터의 `sys.stderr`는 진짜 파일이 아니라** 결과창으로 보내는 가짜 객체입니다.
파일 번호(`fileno`)가 없어서 하위 프로세스에 넘길 수 없습니다.

오류 로그를 **진짜 파일**로 돌려 주면 해결됩니다. 터미널에서 `.py`로 실행할 때는 필요 없습니다.

In [2]:
import functools

import agents.mcp.server as agents_mcp
import mcp.client.stdio as mcp_stdio

MCP_LOG = open("mcp_server.log", "w", encoding="utf-8")
# stdio_client 의 errlog 기본값이 import 시점의 sys.stderr 로 굳어 있다.
# 진짜 파일을 쓰도록 바꿔 끼운다.
agents_mcp.stdio_client = functools.partial(mcp_stdio.stdio_client, errlog=MCP_LOG)

print("MCP 서버 오류 로그 ->", Path("mcp_server.log").resolve())

MCP 서버 오류 로그 -> D:\ai_edu_main\llm_edu\mcp\mcp_server.log


### MCP 응답을 읽는 도우미

MCP 도구의 결과는 **콘텐츠 블록 목록**으로 옵니다. 글자만 꺼내 쓰는 일이 잦으니 함수로 둡니다.

In [3]:
def tool_text(result) -> str:
    """MCP 도구 결과에서 글자만 모은다."""
    return "".join(getattr(block, "text", "") for block in result.content)


def called_tools(run_result) -> list[str]:
    """에이전트가 이번 실행에서 부른 도구 이름."""
    return [getattr(item.raw_item, "name", "?") for item in run_result.new_items
            if item.type == "tool_call_item"]

## Step 1. 샌드박스와 입력 자료

에이전트가 읽고 쓸 **유일한 폴더**를 만듭니다. 여기에 사내 자산 목록을 둡니다.

In [4]:
if SANDBOX.exists():
    shutil.rmtree(SANDBOX)
(SANDBOX / "input").mkdir(parents=True)
(SANDBOX / "output").mkdir()

(SANDBOX / "input" / "assets.md").write_text("""# 사내 자산 (2026-10)
| 제품 | 사용 버전 | 담당 |
|---|---|---|
| Apache Log4j | 2.14.1 | 플랫폼팀 |
| Cisco AnyConnect | 4.9.0 | 인프라팀 |
""", encoding="utf-8")

print("샌드박스 준비:")
for p in sorted(SANDBOX.rglob("*")):
    print("  ", p.relative_to(SANDBOX).as_posix())

샌드박스 준비:
   input
   input/assets.md
   output


## Step 2. MCP 서버 1 — 로컬 리소스 (filesystem)

```python
MCPServerStdio(params={"command": "npx",
                       "args": ["-y", "@modelcontextprotocol/server-filesystem", <폴더>]})
```

**stdio** 방식입니다. SDK가 `npx` 명령으로 서버를 **자식 프로세스로 띄우고**,
표준 입출력으로 JSON-RPC를 주고받습니다. 네트워크를 쓰지 않습니다.

### 샌드박스는 여기서 정해집니다

마지막 인자로 넘긴 폴더가 이 서버가 다룰 수 있는 **전부**입니다.
별도의 샌드박스 장치를 만들 필요가 없습니다 — **서버 설정이 곧 경계**입니다.

In [5]:
fs_server = MCPServerStdio(
    name="filesystem",
    params={"command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", str(SANDBOX)]},
    client_session_timeout_seconds=60,
)

await fs_server.connect()
fs_tools = await fs_server.list_tools()
print(f"도구 {len(fs_tools)}개")
print(" ", [t.name for t in fs_tools])

도구 14개
  ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories']


읽기·쓰기·검색 도구가 한 번에 생겼습니다. **우리가 만든 것이 하나도 없습니다.**
이것이 MCP의 핵심입니다 — 도구를 만드는 대신 **이미 있는 서버를 꽂습니다.**

In [6]:
res = await fs_server.call_tool("read_text_file", {"path": str(SANDBOX / "input" / "assets.md")})
print(tool_text(res))

# 사내 자산 (2026-10)
| 제품 | 사용 버전 | 담당 |
|---|---|---|
| Apache Log4j | 2.14.1 | 플랫폼팀 |
| Cisco AnyConnect | 4.9.0 | 인프라팀 |



## Step 3. MCP 서버 2 — 원격 리소스 (fetch)

```python
MCPServerStdio(params={"command": "uvx", "args": ["mcp-server-fetch"]})
```

**서버는 똑같이 내 PC에서 stdio로 돕니다.** 그런데 이 서버는 인터넷을 봅니다.

| | filesystem | fetch |
|---|---|---|
| 서버 위치 | 내 PC | 내 PC |
| 전송 | stdio | stdio |
| **리소스** | **내 디스크** | **인터넷** |

보안 관점에서 둘은 완전히 다릅니다. filesystem 서버는 데이터가 밖으로 나갈 수 없지만,
fetch 서버는 **외부와 통신합니다.** "로컬 서버라서 안전하다"는 말이 성립하지 않는 이유입니다.

In [7]:
fetch_server = MCPServerStdio(
    name="fetch",
    params={"command": "uvx", "args": ["mcp-server-fetch"]},
    client_session_timeout_seconds=90,
)

await fetch_server.connect()
fetch_tools = await fetch_server.list_tools()
print(f"도구 {len(fetch_tools)}개:", [t.name for t in fetch_tools])

res = await fetch_server.call_tool("fetch", {"url": "https://example.com", "max_length": 400})
print()
print(tool_text(res)[:300])

도구 1개: ['fetch']

Contents of https://example.com/:
<p>Example Domain</p>

This domain is for use in documentation examples without needing permission. This is not a service; avoid relying on it for testing and monitoring purposes.


## Step 4. fetch 로는 안 되는 페이지 — playwright 가 필요한 이유

`fetch`는 HTTP로 HTML을 받아 글자를 추립니다. **자바스크립트를 실행하지 않습니다.**
요즘 공지 페이지는 본문을 JS가 그리는 경우가 많아, 받아 보면 **빈 껍데기**입니다.

직접 확인해 봅니다. 아래 페이지는 본문을 JS가 채웁니다.

In [8]:
SITE = SANDBOX / "site"
SITE.mkdir(exist_ok=True)
(SITE / "advisory.html").write_text("""<!doctype html>
<html lang="ko">
<head><meta charset="utf-8"><title>보안 공지</title></head>
<body>
  <h1>벤더 보안 공지</h1>
  <div id="advisory">불러오는 중...</div>
  <script>
    // 본문을 자바스크립트가 그린다 - HTML 소스에는 완성된 글이 없다
    setTimeout(() => {
      document.getElementById("advisory").innerHTML = `
        <p>영향 제품: Apache Log4j 2.0-beta9 ~ 2.14.1</p>
        <p>CVE: CVE-2021-44228</p>
        <p>CVSS: 10.0 CRITICAL</p>
        <p>조치: 2.17.1 이상으로 업그레이드</p>`;
    }, 300);
  </script>
</body></html>""", encoding="utf-8")

# 작은 웹 서버로 띄운다 (진짜 공지 사이트를 흉내)
class QuietHandler(http.server.SimpleHTTPRequestHandler):
    """접근 로그를 찍지 않는다. 노트북 출력이 지저분해지지 않도록."""

    def log_message(self, *args):
        pass


# 이 셀을 다시 실행해도 괜찮도록, 앞서 띄운 서버가 있으면 먼저 내린다.
# 그냥 두면 "포트는 하나만 사용할 수 있습니다"(WinError 10048) 로 실패한다.
if "httpd" in globals():
    httpd.shutdown()
    httpd.server_close()

handler = functools.partial(QuietHandler, directory=str(SITE))
httpd = socketserver.TCPServer(("127.0.0.1", 0), handler)   # 0 = 빈 포트를 OS 가 골라 준다
site_port = httpd.server_address[1]
threading.Thread(target=httpd.serve_forever, daemon=True).start()

ADVISORY_URL = f"http://127.0.0.1:{site_port}/advisory.html"
print("공지 페이지:", ADVISORY_URL)

공지 페이지: http://127.0.0.1:55317/advisory.html


In [9]:
res = await fetch_server.call_tool("fetch", {"url": ADVISORY_URL})
fetched = tool_text(res)

print("fetch 결과:", len(fetched), "자")
print(fetched)
print()
print("CVE 번호가 들어 있나:", "CVE-2021-44228" in fetched)

fetch 결과: 210 자
Refused to fetch http://127.0.0.1:55317/robots.txt: 127.0.0.1 resolves to 127.0.0.1, which is not a public address. Start the server with --allow-private-ips to allow private, loopback and link-local addresses.

CVE 번호가 들어 있나: False


**본문이 통째로 없습니다.** "불러오는 중..."만 가져왔습니다.

왜 그런지 원본 HTML을 보면 분명해집니다.

In [10]:
res = await fetch_server.call_tool("fetch", {"url": ADVISORY_URL, "raw": True})
raw = tool_text(res)
print("CVE 번호가 원본 HTML에 있나:", "CVE-2021-44228" in raw)
print("어디에 있나 — <script> 안:", "<script" in raw)

CVE 번호가 원본 HTML에 있나: False
어디에 있나 — <script> 안: False


데이터는 HTML 안에 **있습니다.** 다만 `<script>` 안의 코드이고,
**브라우저가 실행해야** 화면의 글이 됩니다. fetch는 실행하지 않으므로 보지 못합니다.

### playwright MCP — 진짜 브라우저를 띄운다

In [11]:
pw_server = MCPServerStdio(
    name="playwright",
    params={"command": "npx",
            "args": ["-y", "@playwright/mcp@latest", "--headless",
                     "--isolated", "--browser", "chromium"]},
    client_session_timeout_seconds=180,
)

await pw_server.connect()
pw_tools = await pw_server.list_tools()
print(f"도구 {len(pw_tools)}개 (일부):", [t.name for t in pw_tools][:8])

도구 25개 (일부): ['browser_close', 'browser_resize', 'browser_console_messages', 'browser_handle_dialog', 'browser_emulate_media', 'browser_evaluate', 'browser_file_upload', 'browser_drop']


In [12]:
await pw_server.call_tool("browser_navigate", {"url": ADVISORY_URL})
# JS 가 본문을 그릴 때까지 기다린다 - 브라우저 자동화에서 가장 흔한 실수가 이 대기다
await pw_server.call_tool("browser_wait_for", {"text": "CVE-2021-44228"})

res = await pw_server.call_tool("browser_evaluate",
                                {"function": "() => document.body.innerText"})
rendered = tool_text(res)

print("playwright 결과:", len(rendered), "자")
print(rendered[:400])

playwright 결과: 222 자
### Result
"벤더 보안 공지\n\n영향 제품: Apache Log4j 2.0-beta9 ~ 2.14.1\n\nCVE: CVE-2021-44228\n\nCVSS: 10.0 CRITICAL\n\n조치: 2.17.1 이상으로 업그레이드"
### Ran Playwright code
```js
await page.evaluate('() => document.body.innerText');
```


| | 길이 | CVE 번호 | CVSS |
|---|---|---|---|
| `fetch` | 84자 | 없음 | 없음 |
| `playwright` | 220자 남짓 | **있음** | **있음** |

같은 주소인데 결과가 다릅니다. **무엇으로 가져오느냐가 내용을 바꿉니다.**

> `browser_wait_for` 를 빼면 실패합니다. 본문이 그려지기 전에 읽어 버리기 때문입니다.
> 브라우저 자동화에서는 **"언제 읽을 것인가"가 "무엇을 읽을 것인가"만큼 중요합니다.**

> playwright는 느리고 무겁습니다(브라우저를 띄웁니다). **fetch로 되는 페이지는 fetch로** 하고,
> 안 될 때만 playwright를 쓰세요.

## Step 5. MCP 클라이언트 1 — 원격 서버에 접속

앞의 둘은 SDK가 **서버를 직접 띄웠습니다**(stdio). 이번에는 다릅니다.
이미 **다른 곳에서 돌고 있는 서버**에 HTTP로 접속합니다.

사내 CVE 조회 서비스가 그런 경우입니다 — 팀이 공유하므로 각자 띄우지 않습니다.

### 먼저 그 서버를 만듭니다

In [13]:
SERVER_CODE = '''"""사내 CVE 조회 MCP 서버."""
import sys

from mcp.server.mcpserver import MCPServer

CVE_DB = {
    "CVE-2021-44228": {"product": "Apache Log4j", "cvss": 10.0,
                       "affected": "2.0-beta9 ~ 2.14.1", "fixed_in": "2.17.1"},
    "CVE-2024-20337": {"product": "Cisco AnyConnect", "cvss": 8.2,
                       "affected": "~ 4.10.04065", "fixed_in": "4.10.07061"},
}

server = MCPServer(name="cve-lookup")


@server.tool()
def lookup_cve(cve_id: str) -> str:
    """사내 CVE 데이터베이스에서 CVE 정보를 조회한다."""
    info = CVE_DB.get(cve_id.upper())
    return str(info) if info else f"{cve_id}: 등록되지 않음"


@server.tool()
def list_cves() -> str:
    """등록된 CVE 번호를 모두 돌려준다."""
    return ", ".join(CVE_DB)


if __name__ == "__main__":
    port = int(sys.argv[1]) if len(sys.argv) > 1 else 8931
    server.run(transport="streamable-http", host="127.0.0.1", port=port)
'''

SERVER_FILE = Path("cve_server.py")
SERVER_FILE.write_text(SERVER_CODE, encoding="utf-8")
print(SERVER_CODE[:430])

"""사내 CVE 조회 MCP 서버."""
import sys

from mcp.server.mcpserver import MCPServer

CVE_DB = {
    "CVE-2021-44228": {"product": "Apache Log4j", "cvss": 10.0,
                       "affected": "2.0-beta9 ~ 2.14.1", "fixed_in": "2.17.1"},
    "CVE-2024-20337": {"product": "Cisco AnyConnect", "cvss": 8.2,
                       "affected": "~ 4.10.04065", "fixed_in": "4.10.07061"},
}

server = MCPServer(name="cve-lookup")


@server


`MCPServer` + `@server.tool()` 두 가지면 서버가 됩니다.
**도구 함수는 평범한 파이썬 함수**이고, 독스트링이 그대로 도구 설명이 됩니다.

마지막 줄이 전송 방식입니다.

```python
server.run(transport="streamable-http", host="127.0.0.1", port=8931)
```

`transport="stdio"` 로 바꾸면 앞의 두 서버처럼 자식 프로세스형이 됩니다.
**같은 코드가 전송 방식만 바꿔 로컬형도 되고 원격형도 됩니다.**

In [14]:
import socket
import urllib.request


def free_port() -> int:
    """비어 있는 포트를 하나 얻는다."""
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


# 이 셀을 다시 실행해도 괜찮도록, 앞서 띄운 서버가 있으면 먼저 내린다
if "proc" in globals():
    proc.terminate()
    proc.wait(timeout=10)

cve_port = free_port()
# 별도 프로세스로 띄운다 (실제로는 다른 서버에서 이미 돌고 있을 것이다)
proc = subprocess.Popen([sys.executable, str(SERVER_FILE), str(cve_port)],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

CVE_URL = f"http://127.0.0.1:{cve_port}/mcp"
for _ in range(40):                      # 뜰 때까지 기다린다
    try:
        urllib.request.urlopen(CVE_URL, timeout=1)
        break
    except urllib.error.HTTPError:       # 응답이 왔다는 뜻 - 기동 완료
        break
    except Exception:
        time.sleep(0.5)
print(f"서버 기동 완료 (pid {proc.pid}, {CVE_URL})")

서버 기동 완료 (pid 10980, http://127.0.0.1:51224/mcp)


In [15]:
cve_server = MCPServerStreamableHttp(
    name="cve-lookup",
    params={"url": CVE_URL},                          # <- 주소만 바꾸면 진짜 원격
    client_session_timeout_seconds=60,
)

await cve_server.connect()
cve_tools = await cve_server.list_tools()
print(f"도구 {len(cve_tools)}개:", [t.name for t in cve_tools])

res = await cve_server.call_tool("lookup_cve", {"cve_id": "CVE-2021-44228"})
print()
print(tool_text(res))

도구 2개: ['lookup_cve', 'list_cves']

{'product': 'Apache Log4j', 'cvss': 10.0, 'affected': '2.0-beta9 ~ 2.14.1', 'fixed_in': '2.17.1'}


**클라이언트 코드에 "로컬"이라는 표시가 없습니다.** `url` 이 전부입니다.
그 주소를 사내 서버나 클라우드 주소로 바꾸면 **코드 수정 없이** 원격이 됩니다.

> 지금은 `127.0.0.1` 이라 같은 PC지만, **연결 방식은 원격과 완전히 동일합니다.**
> stdio 서버는 주소가 아예 없다는 점과 비교해 보세요.

> 연결할 때 pydantic 경고가 길게 나올 수 있습니다. 클라이언트가 보낸 신형 요청을
> 서버 라이브러리가 모르는 경우로, **동작에는 지장이 없습니다.** MCP는 아직 버전이
> 빠르게 올라가는 규격이라 이런 불일치를 자주 만납니다.

## Step 6. 셋을 모두 붙인 에이전트

```python
Agent(..., mcp_servers=[fs_server, fetch_server, cve_server])
```

`mcp_servers` 에 넣기만 하면 **세 서버의 도구가 모두** 에이전트의 도구가 됩니다.
어느 서버의 도구인지 에이전트는 신경 쓰지 않습니다.

### trace 로 전체를 하나로 묶기

`with trace("이름")` 안에서 실행하면 LLM 호출과 **MCP 도구 호출까지** 하나의 기록으로 묶입니다.
<https://platform.openai.com/traces> 에서 볼 수 있습니다.

In [16]:
analyst = Agent(
    name="security_analyst",
    instructions=f"""너는 보안 분석가다.
- 사내 자산 목록은 filesystem 도구로 {SANDBOX / "input"} 에서 읽는다.
- 벤더 공지는 fetch 또는 playwright 로 가져온다.
- CVE 상세는 cve-lookup 도구로 조회한다. 기억으로 답하지 마라.
- 최종 보고서는 filesystem 도구로 {SANDBOX / "output"} 아래에 저장한다.
한국어로 간결하게 답하라.""",
    model=MODEL,
    mcp_servers=[fs_server, fetch_server, cve_server],
)

print("에이전트 준비 — MCP 서버", len(analyst.mcp_servers), "개 연결")

에이전트 준비 — MCP 서버 3 개 연결


In [17]:
task = (f"input/assets.md 의 자산 목록을 읽고, 각 제품과 관련된 CVE 를 cve-lookup 으로 조회해 "
        f"사내 사용 버전이 영향을 받는지 판정해줘. "
        f"결과를 output/report.md 로 저장하고, 무엇을 어떻게 확인했는지도 적어줘.")

with trace("mcp_security_report"):
    result = await Runner.run(analyst, task, max_turns=20)

print("부른 도구:", called_tools(result))
print()
print(result.final_output[:400])

부른 도구: ['read_text_file', 'lookup_cve', 'lookup_cve', 'lookup_cve', 'lookup_cve', 'lookup_cve', 'write_file']

보고서를 `output/report.md`에 저장했습니다.

- Apache Log4j 2.14.1: 영향 있음
- Cisco AnyConnect 4.9.0: 현재 조회 기준으로 판정 불가

확인 방법도 함께 기록했습니다.


도구 이름만 봐도 **어느 서버를 거쳤는지** 보입니다 —
`read_text_file`(서버 1) · `lookup_cve`(클라이언트 1) · `write_file`(서버 1).

In [18]:
print((SANDBOX / "output" / "report.md").read_text(encoding="utf-8")[:700])

# CVE 점검 보고서

## 점검 대상
- Apache Log4j 2.14.1
- Cisco AnyConnect 4.9.0

## 확인 방법
1. `input/assets.md`에서 사내 사용 제품과 버전을 확인함.
2. 각 제품 관련 CVE를 `cve-lookup`으로 조회함.
3. 조회 결과의 영향 버전과 사내 버전을 비교해 영향 여부를 판정함.

## 결과

### Apache Log4j 2.14.1
- 조회 CVE: CVE-2021-44228
- 조회 결과: 영향 버전 `2.0-beta9 ~ 2.14.1`, 수정 버전 `2.17.1`, CVSS 10.0
- 판정: **영향 있음**
- 사유: 사용 버전 2.14.1이 영향 범위에 포함됨.

### Cisco AnyConnect 4.9.0
- 조회 CVE: 관련 CVE를 조회했으나 등록 결과 없음
- 판정: **판정 불가 / 추가 확인 필요**
- 사유: 현재 조회한 CVE 목록에서는 해당 제품에 대한 등록 정보를 확인하지 못함.

## 요약
- 영향 있음: Apache Log4j 2.14.1
- 추가 확인 필요: Cisco AnyConnect 4.9.0



## Step 7. 샌드박스 경계 확인

filesystem 서버에 넘긴 폴더가 정말 경계인지 **직접 시험해 봅니다.**

In [19]:
probe = Agent(
    name="boundary_probe",
    instructions="filesystem 도구로 요청받은 파일을 읽어라. 실패하면 실패 사유를 그대로 전해라.",
    model=MODEL,
    mcp_servers=[fs_server],
)

with trace("sandbox_boundary"):
    out = await Runner.run(probe, r"C:\Windows\win.ini 파일을 읽어서 첫 줄을 알려줘.")

print(out.final_output[:300])

읽을 수 없습니다. 허용된 디렉터리에 `C:\Windows\win.ini`가 없어서 접근 불가입니다.

허용된 경로:
`D:\ai_edu_main\llm_edu\mcp\sandbox_agent`


읽지 못합니다. 서버를 띄울 때 넘긴 폴더 밖은 **서버가 거부**합니다.

중요한 점은 거부하는 주체가 **에이전트도, 모델도 아니라 MCP 서버**라는 것입니다.
모델이 아무리 설득당해도(프롬프트 인젝션) 서버가 막습니다.
**지시문으로 만든 규칙과 구조로 만든 경계는 다릅니다.**

## Step 8. 정리

In [20]:
# 서버 연결을 모두 정리한다. 자식 프로세스가 남지 않도록
for server in (fs_server, fetch_server, pw_server, cve_server):
    try:
        await server.cleanup()
    except Exception as e:
        print(f"{server.name} 정리 중: {type(e).__name__}")

proc.terminate()
httpd.shutdown()
print("정리 완료")

정리 완료


### 세 가지 연결 방식

| | MCP 서버 1 | MCP 서버 2 | MCP 클라이언트 1 |
|---|---|---|---|
| 예시 | filesystem | fetch · playwright | 사내 CVE 조회 |
| 클래스 | `MCPServerStdio` | `MCPServerStdio` | `MCPServerStreamableHttp` |
| 서버를 누가 띄우나 | **SDK가 자식 프로세스로** | SDK가 자식 프로세스로 | **이미 떠 있음** |
| 전송 | 표준 입출력 | 표준 입출력 | HTTP |
| 리소스 | **내 디스크** | **인터넷** | 그 서버가 가진 것 |
| 설정에 필요한 것 | 실행 명령 | 실행 명령 | **URL** |

### 고를 때 기준

- **내 파일을 다룬다** → stdio + 폴더를 좁게 지정. 데이터가 밖으로 나가지 않습니다.
- **웹 문서가 필요하다** → fetch 먼저. 안 되면 playwright.
- **팀이 함께 쓴다** → HTTP 서버로 한 번 띄우고 모두 접속.

### 이 노트북에서 기억할 것

| | |
|---|---|
| 서버 위치 ≠ 리소스 위치 | fetch 서버는 로컬이지만 인터넷을 봅니다 |
| 샌드박스는 서버 설정 | filesystem 서버에 넘긴 폴더가 곧 경계 |
| 경계는 구조로 | 지시문이 아니라 서버가 거부합니다 |
| fetch는 JS를 실행 안 함 | 같은 주소라도 결과가 다릅니다 |
| 브라우저는 기다림이 핵심 | `browser_wait_for` 없으면 빈 화면을 읽습니다 |
| 전송 방식은 바꿔 끼울 수 있다 | 같은 서버 코드가 stdio도 되고 HTTP도 됩니다 |

### 더 해볼 것

1. filesystem 서버의 폴더를 `sandbox/input` 으로 좁혀 보세요. 보고서 저장이 실패합니다 — 최소 권한의 대가입니다.
2. `cve_server.py` 의 `transport` 를 `"stdio"` 로 바꾸고 `MCPServerStdio` 로 연결해 보세요. 도구는 똑같습니다.
3. `browser_wait_for` 줄을 지우고 Step 4를 다시 실행해 보세요.
4. 에이전트에게 fetch 로 Step 4의 공지를 읽게 하고, 판정이 어떻게 틀어지는지 보세요.